<a href="https://colab.research.google.com/github/Plabon-Basak/FlyRank_AI/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Plabon-Basak/FlyRank_AI/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
!git clone https://github.com/Plabon-Basak/FlyRank_AI.git
%cd FlyRank_AI

Cloning into 'FlyRank_AI'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 127 (delta 39), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 1.87 MiB | 10.39 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/FlyRank_AI


## 1. My lane as an ML task (type)

I am framing my lane as a ranking/scoring problem. The goal is to assign each content page a priority score so that a content or SEO reviewer can decide which pages should be reviewed first. The output is not an automatic decision to change a page; it is a ranked review queue. Ranking/scoring fits this decision because the practical question is "which pages should be reviewed first?" rather than only "is this page declining?"

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Task type: Ranking / Scoring")
print("Decision grain: one content page")

Task type: Ranking / Scoring
Decision grain: one content page


## 2. Target or proxy

The initial proxy target is whether a page is observed as declining in the starter snapshot. I define the proxy as is_declining_label = 1 when trend_direction == "down" and 0 otherwise. This is an observed snapshot label, not a future outcome, so it should not be treated as proof that a page will decline later. For a stronger later model, I would construct the target from a future time window and use only earlier information as features.

In [5]:
import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Proxy target distribution:")
print(df["is_declining_label"].value_counts().sort_index())

print("\nProxy target rate:")
print(df["is_declining_label"].mean())

Proxy target distribution:
is_declining_label
0    13738
1    16262
Name: count, dtype: int64

Proxy target rate:
0.5420666666666667


## 3. Success metric

I will use Precision@100 as the initial success metric. It measures how many of the top 100 pages in the ranked review queue are observed as declining. This matches the practical constraint that a reviewer has limited time and may only review a small number of pages first. A higher Precision@100 means the top of the queue contains more pages with the observed declining signal.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Success metric: Precision@100")
print("Review queue size: 100 pages")

Success metric: Precision@100
Review queue size: 100 pages


## 4. The unit of analysis, as a real dataframe

The unit of analysis is one content page. Each row represents one pseudonymized content item, with page-level search, engagement, age, and trend-related measurements. The decision is therefore made at the content-page level: which individual pages should be reviewed first?

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane_columns = [
    "content_id",
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "engagement_rate",
    "avg_position",
    "trend_direction",
    "is_declining_label"
]

lane_df = df[lane_columns].copy()

print("One row = one content page")
print("Rows:", len(lane_df))
print("Unique content IDs:", lane_df["content_id"].nunique())

display(lane_df.head(10))

One row = one content page
Rows: 30000
Unique content IDs: 30000


,content_id,content_type,content_age_days,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,ctr,engagement_rate,avg_position,trend_direction,is_declining_label
0,content_304f48230142,keyword article,187,20,3803,29,17,0.76,5.88,10.6,down,1
1,content_a1fb4e703a9e,keyword article,445,25,15320,7,9,0.05,0.00,20.3,down,1
2,content_9aa793d4d895,keyword article,141,20,12581,11,11,0.09,0.00,36.5,down,1
3,content_331d6c4de07b,keyword article,463,22,11751,58,78,0.49,1.28,6.2,stable,0
4,content_d99b7a2d90ca,keyword article,263,14,19140,24,145,0.13,0.00,44.0,down,1
5,content_d4084a4bc775,keyword article,147,20,3970,1,5,0.03,0.00,8.5,down,1
6,content_9a34b442b552,keyword article,90,20,20,0,1,0.00,0.00,7.0,down,1
7,content_a63219c6e95a,keyword article,445,22,1724,1,28,0.06,3.57,21.2,stable,0
8,content_5e6c160719bc,keyword article,90,20,32574,29,68,0.09,5.88,46.0,down,1
9,content_c27558df2b0c,keyword article,257,104,1240,2,3,0.16,0.00,4.9,down,1


## 5. Why ML beats a fixed rule here

A fixed rule could flag pages using one threshold, such as pages older than a certain number of days or pages with a low CTR. However, the review priority depends on several signals together, including content age, time since the last update, impressions, clicks, sessions, CTR, engagement, average position, content type, and observed trend. These signals can interact in ways that are difficult to capture with one simple threshold. ML could learn combinations of these measured signals and produce a ranked review queue, while the reviewer still makes the final content decision.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Potential ranking signals:")
print([
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "engagement_rate",
    "avg_position",
    "content_type"
])

print("\nML output: a priority score used to rank pages for human review.")

Potential ranking signals:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'engagement_rate', 'avg_position', 'content_type']

ML output: a priority score used to rank pages for human review.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.